In [1]:
import os, glob, re, random, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from scipy import stats
from PIL import Image

warnings.filterwarnings('ignore')

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT    = '/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset'
BASE_H  = 176
BASE_W  = 512
N_CLASSES = 6
SEED    = 42
EPOCHS  = 50
BATCH   = 64
LR      = 1e-3
K_FOLDS = 5
GAP     = 8
WIDTH   = 32
OC      = 48
USE_NOISE = False
TRAIN_SNR_RANGE = (-5.0, 20.0)
TRAIN_NOISE_P   = 0.85

CLASS_MAP = {
    '3_long_blade_rotor':     '3_long_blade_rotor',
    '3_short_blade_rotor_1':  '3_short_blade_rotor',
    '3_short_blade_rotor_2':  '3_short_blade_rotor',
    'Bird':                   'Bird',
    'Bird+mini-helicopter_1': 'Bird+mini-helicopter',
    'Bird+mini-helicopter_2': 'Bird+mini-helicopter',
    'RC plane_1':             'RC_plane',
    'RC plane_2':             'RC_plane',
    'drone_1':                'drone',
    'drone_2':                'drone',
}
CLASSES = sorted(set(CLASS_MAP.values()))
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}


def numeric_key(p):
    nums = re.findall(r'\d+', os.path.basename(p))
    return int(nums[0]) if nums else 0


def list_images(folder):
    exts = ('*.png','*.jpg','*.jpeg','*.PNG','*.JPG','*.JPEG')
    out = []
    for e in exts:
        out += glob.glob(os.path.join(folder, '**', e), recursive=True)
    return out


def autocrop_resize(path):
    img = Image.open(path).convert('L')
    arr = np.asarray(img, dtype=np.float32)
    mask = arr < 240
    if mask.any():
        rows = np.where(mask.any(axis=1))[0]
        cols = np.where(mask.any(axis=0))[0]
        arr = arr[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1]
    return np.asarray(
        Image.fromarray(arr.astype(np.uint8)).resize((BASE_W, BASE_H), Image.BILINEAR),
        dtype=np.uint8)


print('Enumerating files in filename (recording) order...')
paths, labels = [], []
for top in sorted(os.listdir(ROOT)):
    if top in CLASS_MAP:
        c = CLS2IDX[CLASS_MAP[top]]
        for f in sorted(list_images(os.path.join(ROOT, top)), key=lambda p: (numeric_key(p), p)):
            paths.append(f); labels.append(c)
labels = np.array(labels, dtype=np.int64)
N = len(paths)
print('Total images:', N)

print('Loading images...')
bdata = np.zeros((N, BASE_H, BASE_W), dtype=np.uint8)
for i, p in enumerate(paths):
    bdata[i] = autocrop_resize(p)
print('Loaded:', bdata.shape)

idx_all = np.arange(N)


def trim(a, g):
    return a[g:-g] if len(a) > 2 * g + 1 else a


def make_folds(K, gap):
    cb = {c: np.array_split(idx_all[labels == c], K) for c in range(N_CLASSES)}
    folds = []
    for k in range(K):
        tr, va, te = [], [], []
        for c in range(N_CLASSES):
            bl = cb[c]
            te.append(trim(bl[k], gap)); va.append(trim(bl[(k + 1) % K], gap))
            for j in range(K):
                if j != k and j != (k + 1) % K:
                    tr.append(trim(bl[j], gap))
        folds.append((np.concatenate(tr), np.concatenate(va), np.concatenate(te)))
    return folds


FOLDS = make_folds(K_FOLDS, GAP)
print(f'Honest folds K={K_FOLDS}, GAP={GAP}, mode={"NOISE-AUG" if USE_NOISE else "CLEAN"}')


def add_noise(img, snr_db):
    if snr_db is None:
        return img
    sig = float(img.var()) + 1e-8
    npow = sig / (10 ** (snr_db / 10.0))
    noise = np.random.randn(*img.shape).astype(np.float32) * np.sqrt(npow)
    return np.clip(img + noise, 0.0, 1.0).astype(np.float32)


class ConvBNSiLU(nn.Module):
    def __init__(self, ci, co, k=3, s=1):
        super().__init__()
        self.b = nn.Sequential(nn.Conv2d(ci, co, k, s, k // 2, bias=False),
                               nn.BatchNorm2d(co), nn.SiLU(inplace=True))
    def forward(self, x): return self.b(x)


class DSConv2d(nn.Module):
    def __init__(self, ci, co, s=1):
        super().__init__()
        self.dw = nn.Conv2d(ci, ci, 3, s, 1, groups=ci, bias=False)
        self.pw = nn.Conv2d(ci, co, 1, bias=False)
        self.bn = nn.BatchNorm2d(co); self.act = nn.SiLU(inplace=True)
    def forward(self, x): return self.act(self.bn(self.pw(self.dw(x))))


class ResDS2d(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.b = nn.Sequential(DSConv2d(ch, ch, 1), DSConv2d(ch, ch, 1))
    def forward(self, x): return x + self.b(x)


def cvd_parts(x, use_phase):
    with torch.amp.autocast('cuda', enabled=False):
        xf = x.float()
        c = torch.fft.rfft(xf, dim=3)
        mag = torch.log1p(c.abs())
        if not use_phase:
            return mag
        ph = torch.atan2(c.imag, c.real + 1e-8)
        return torch.cat([mag, torch.sin(ph), torch.cos(ph)], dim=1)


class MultiResCVD(nn.Module):
    def __init__(self, splits, use_phase):
        super().__init__()
        self.splits = splits; self.use_phase = use_phase
    def forward(self, x):
        B, C, Hh, Ww = x.shape
        feats = []
        for nsp in self.splits:
            seg = Ww // nsp
            segouts = []
            for i in range(nsp):
                segouts.append(cvd_parts(x[:, :, :, i * seg:(i + 1) * seg], self.use_phase))
            ref = segouts[0].shape[-1]
            segouts = [F.adaptive_avg_pool2d(o, (Hh, ref)) for o in segouts]
            feats.append(torch.stack(segouts, 0).mean(0))
        ref = feats[0].shape[-1]
        feats = [F.adaptive_avg_pool2d(f, (Hh, ref)) for f in feats]
        return torch.cat(feats, dim=1)


class CadenceAttn(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.q = nn.Conv2d(ch, max(ch // 4, 4), 1)
        self.k = nn.Conv2d(ch, max(ch // 4, 4), 1)
        self.proj = nn.Conv2d(ch, ch, 1)
        self.bn = nn.BatchNorm2d(ch)
    def forward(self, x):
        B, C, Hh, Wc = x.shape
        q = self.q(x).mean(2); k = self.k(x).mean(2)
        a = torch.softmax(torch.bmm(q.transpose(1, 2), k) / (q.size(1) ** 0.5), dim=-1)
        v = x.mean(2).transpose(1, 2)
        o = torch.bmm(a, v).transpose(1, 2).unsqueeze(2)
        return self.bn(x + self.proj(o.expand(-1, -1, Hh, -1)))


class CadenceNet(nn.Module):
    def __init__(self, use_phase=True, splits=(1, 2), use_attn=True, depth=2,
                 width=WIDTH, oc=OC, n=N_CLASSES):
        super().__init__()
        self.stem = nn.Sequential(ConvBNSiLU(1, 16, 3, 2), ConvBNSiLU(16, width, 3, 2), ResDS2d(width))
        self.cvd = MultiResCVD(splits, use_phase)
        mult = (3 if use_phase else 1) * len(splits)
        self.reduce = ConvBNSiLU(width * mult, oc, 1)
        self.use_attn = use_attn
        if use_attn:
            self.attn = CadenceAttn(oc)
        self.body = nn.Sequential(DSConv2d(oc, oc, 2), *[ResDS2d(oc) for _ in range(depth)],
                                  DSConv2d(oc, oc, 2))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fuse = nn.Sequential(nn.Linear(oc, oc), nn.BatchNorm1d(oc),
                                  nn.SiLU(inplace=True), nn.Dropout(0.3))
        self.cls = nn.Linear(oc, n)
    def forward(self, x):
        f = self.stem(x)
        c = self.reduce(self.cvd(f))
        if self.use_attn:
            c = self.attn(c)
        return self.cls(self.fuse(self.gap(self.body(c)).flatten(1)))


class DS(Dataset):
    def __init__(self, idx, train):
        self.idx = idx; self.train = train
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        img = bdata[j].astype(np.float32) / 255.0
        if self.train:
            if random.random() < 0.5:
                img = img[:, ::-1].copy()
            if random.random() < 0.5:
                img = np.clip(img * random.uniform(0.9, 1.1), 0.0, 1.0).astype(np.float32)
            if USE_NOISE and random.random() < TRAIN_NOISE_P:
                img = add_noise(img, random.uniform(*TRAIN_SNR_RANGE))
            if random.random() < 0.3:
                h0 = random.randint(0, BASE_H - 20)
                img[h0:h0 + random.randint(5, 20), :] = 0.0
            if random.random() < 0.3:
                w0 = random.randint(0, BASE_W - 25)
                img[:, w0:w0 + random.randint(5, 25)] = 0.0
        return torch.from_numpy(img).float().unsqueeze(0), int(labels[j])


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def train_eval_fold(build_fn, tr_idx, va_idx, te_idx):
    set_seed(SEED)
    m = build_fn().to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-2)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    scl = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
    tl = DataLoader(DS(tr_idx, True), batch_size=BATCH, shuffle=True,
                    num_workers=2, pin_memory=True, drop_last=True)
    vl = DataLoader(DS(va_idx, False), batch_size=BATCH, shuffle=False,
                    num_workers=2, pin_memory=True)
    best, best_state = 0.0, None
    for ep in range(EPOCHS):
        m.train()
        for x, y in tl:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad()
            with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
                loss = crit(m(x), y)
            scl.scale(loss).backward(); scl.step(opt); scl.update()
        sch.step()
        m.eval(); c = t = 0
        with torch.no_grad():
            for x, y in vl:
                x, y = x.to(DEVICE), y.to(DEVICE)
                c += (m(x).argmax(1) == y).sum().item(); t += y.size(0)
        if c / t >= best:
            best = c / t
            best_state = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
    m.load_state_dict(best_state); m.eval()
    tel = DataLoader(DS(te_idx, False), batch_size=BATCH, shuffle=False,
                     num_workers=2, pin_memory=True)
    yt, yp = [], []
    with torch.no_grad():
        for x, y in tel:
            yp.append(m(x.to(DEVICE)).argmax(1).cpu().numpy()); yt.append(y.numpy())
    del m
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return accuracy_score(np.concatenate(yt), np.concatenate(yp)), np.concatenate(yp), np.concatenate(yt)


def run_config(build_fn):
    accs, pool_yp, pool_yt = [], [], []
    for (tr, va, te) in FOLDS:
        a, yp, yt = train_eval_fold(build_fn, tr, va, te)
        accs.append(a); pool_yp.append(yp); pool_yt.append(yt)
    accs = np.array(accs)
    ci = stats.t.interval(0.95, len(accs) - 1, loc=accs.mean(), scale=stats.sem(accs))
    return dict(accs=accs, mean=accs.mean(), std=accs.std(ddof=1), ci=ci,
                params=sum(p.numel() for p in build_fn().parameters()),
                pool_yp=np.concatenate(pool_yp), pool_yt=np.concatenate(pool_yt))


def mcnemar(yp_a, yp_b, yt):
    ca = (yp_a == yt); cb = (yp_b == yt)
    b = ((~ca) & cb).sum(); c = (ca & (~cb)).sum()
    if b + c == 0:
        return 1.0
    chi2 = (abs(b - c) - 1) ** 2 / (b + c)
    return float(1 - stats.chi2.cdf(chi2, df=1))


VARIANTS = [
    ('base_cvd',         dict(use_phase=False, splits=(1,),  use_attn=False)),
    ('phase',            dict(use_phase=True,  splits=(1,),  use_attn=False)),
    ('multires',         dict(use_phase=False, splits=(1, 2), use_attn=False)),
    ('cad_attn',         dict(use_phase=False, splits=(1,),  use_attn=True)),
    ('phase+multires',   dict(use_phase=True,  splits=(1, 2), use_attn=False)),
    ('phase+attn',       dict(use_phase=True,  splits=(1,),  use_attn=True)),
    ('multires+attn',    dict(use_phase=False, splits=(1, 2), use_attn=True)),
    ('full',             dict(use_phase=True,  splits=(1, 2), use_attn=True)),
]

print('\n' + '=' * 90)
print(f'STAGE 1 - CADENCE MODULE ABLATION (depth=2, {"NOISE" if USE_NOISE else "CLEAN"})')
print('=' * 90)
s1, cfg = {}, {}
for name, kw in VARIANTS:
    print(f'\nRunning {name} ...')
    r = run_config(lambda k=kw: CadenceNet(depth=2, **k))
    s1[name] = r; cfg[name] = kw
    print(f'  folds {[f"{a*100:.2f}" for a in r["accs"]]}  '
          f'mean {r["mean"]*100:.2f}% +/- {r["std"]*100:.2f}  params {r["params"]:,}')

ref = s1['base_cvd']
print('\n' + '-' * 90)
print(f"{'Variant':<18}{'Mean%':>8}{'+/-Std':>8}{'CI95_lo':>9}{'CI95_hi':>9}{'Params':>9}{'McNemar_p':>11}")
print('-' * 90)
for name, _ in VARIANTS:
    r = s1[name]
    p = 1.0 if name == 'base_cvd' else mcnemar(r['pool_yp'], ref['pool_yp'], ref['pool_yt'])
    r['mcnemar_p'] = p
    star = '*' if (p < 0.05 and r['mean'] > ref['mean']) else ''
    print(f"{name:<18}{r['mean']*100:>8.2f}{r['std']*100:>8.2f}"
          f"{r['ci'][0]*100:>9.2f}{r['ci'][1]*100:>9.2f}{r['params']:>9,}{p:>11.4f}{star}")
print('* = significantly beats base_cvd (McNemar p<0.05 AND higher mean)')

cand = [(n, s1[n]) for n, _ in VARIANTS if s1[n]['mean'] >= max(v['mean'] for v in s1.values()) - 1e-9]
best_overall = max(s1.values(), key=lambda v: v['mean'])
win_name = max(s1, key=lambda n: s1[n]['mean'])
win_kw = cfg[win_name]
print(f'\nSTAGE 1 WINNER (highest mean): {win_name}  {win_kw}')

print('\n' + '=' * 90)
print(f'STAGE 2 - DEPTH SWEEP ON WINNER ({win_name})')
print('=' * 90)
DEPTHS = [1, 2, 3, 4]
s2 = {}
for d in DEPTHS:
    print(f'\nDepth {d} ...')
    r = run_config(lambda dd=d: CadenceNet(depth=dd, **win_kw))
    s2[d] = r
    print(f'  mean {r["mean"]*100:.2f}% +/- {r["std"]*100:.2f}  '
          f'CI[{r["ci"][0]*100:.2f},{r["ci"][1]*100:.2f}]  params {r["params"]:,}')

print('\n' + '-' * 70)
print(f"{'Depth':>6}{'Mean%':>9}{'+/-Std':>8}{'CI95_lo':>9}{'CI95_hi':>9}{'Params':>10}")
print('-' * 70)
for d in DEPTHS:
    r = s2[d]
    print(f"{d:>6}{r['mean']*100:>9.2f}{r['std']*100:>8.2f}"
          f"{r['ci'][0]*100:>9.2f}{r['ci'][1]*100:>9.2f}{r['params']:>10,}")

best_d = max(DEPTHS, key=lambda d: s2[d]['mean'])

print('\n' + '=' * 90)
print('FINAL SELECTION')
print('=' * 90)
print(f'  Mode         : {"noise-augmented" if USE_NOISE else "clean"} honest folds')
print(f'  Modules      : {win_name}  {win_kw}')
print(f'  Best depth   : {best_d}')
print(f'  Honest acc   : {s2[best_d]["mean"]*100:.2f}% +/- {s2[best_d]["std"]*100:.2f}')
print(f'  Params       : {s2[best_d]["params"]:,}')

tag = 'noise' if USE_NOISE else 'clean'
np.savez(os.path.join('/kaggle/working', f'cadence_modules_{tag}.npz'),
         winner=win_name, best_depth=best_d,
         s1_means={k: float(v['mean']) for k, v in s1.items()},
         s2_means={d: float(s2[d]['mean']) for d in DEPTHS})
print(f'\nSaved cadence_modules_{tag}.npz')

Enumerating files in filename (recording) order...
Total images: 4849
Loading images...
Loaded: (4849, 176, 512)
Honest folds K=5, GAP=8, mode=CLEAN

STAGE 1 - CADENCE MODULE ABLATION (depth=2, CLEAN)

Running base_cvd ...
  folds ['83.07', '89.59', '91.76', '93.71', '92.21']  mean 90.07% +/- 4.18  params 28,966

Running phase ...
  folds ['79.52', '91.53', '91.65', '92.79', '92.67']  mean 89.63% +/- 5.68  params 32,038

Running multires ...
  folds ['83.07', '86.73', '91.53', '93.71', '93.36']  mean 89.68% +/- 4.63  params 30,502

Running cad_attn ...
  folds ['80.09', '90.85', '91.53', '94.74', '93.47']  mean 90.14% +/- 5.82  params 32,590

Running phase+multires ...
  folds ['81.24', '91.19', '91.53', '89.02', '92.90']  mean 89.17% +/- 4.65  params 36,646

Running phase+attn ...
  folds ['82.04', '90.16', '93.71', '94.85', '92.90']  mean 90.73% +/- 5.16  params 35,662

Running multires+attn ...
  folds ['83.41', '95.77', '93.94', '95.31', '91.07']  mean 91.90% +/- 5.09  params 34,12